Este cuaderno, tiene el fin de presentar una metodología de calculo para diseñar un inductor para el circuito sintonizado. Para ello, se nos presentan las siguientes especificaciones:
 * Frecuencia de Resonancia $f_{o} = 18 [MHz]$
 * Impedancia de Entrada: $Z_{in} = 50 [\Omega]$
 * Impedancia de Salida: $Z_{o} = 1 [k\Omega]$
 * Factor de Calidad: $Q_{c} = 10 [%]$

De estas especificaciones, determinaremos las características físicas del inductor, tales como:
 * Numero de Espiras ($N$).
 * Longitud del Conductor ($l$).
 * Diametro del Conductor ($D$).
 * Relación de Aspecto ($\frac{l}{D}$).
 * Separación entre Espiras ($S$).

Consecuentemente, tendremos los parámetros de salida y caracteristicas circuitales del inductor, siendo estas:
 * Inductacia ($L$).
 * Factor de Calidad Descargado o del inductor ($Q_{D}$).
 * Resistencia de Pérdida ($R_{P}$).

Finalmente, sabiendo estos valores, podemos plantear un sistema de ecuaciones donde sabiendo el valor de $L$ y de $f_{o}$, despejaremos los valores de los capacitores: $C_{1}$, $C_{2}$, $C_{3}$ y $C_{4}$.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from scipy.optimize import fsolve
from scipy.optimize import least_squares

## 1- Defino la frecuencia de resonancia
f_o_mhz = 18                  # Especificación de frecuencia. 
f_o = f_o_mhz * 1e6           # Conversión a [Hz].
BW = f_o_mhz * 0.1            # Definición del ancho de banda (BW) en [MHz].



## 2- Defino los posibles parámetros físicos del Inductor
# 2.1- Se realizo un arreglo con los valores posibles de los Diametros de cobre esmaltados (d) para su posterior análisis (0.08 - 0.23)cm.
Diametros_cm = np.zeros(16)
for i in range(16):
    Diametros_cm[i] = 0.08+(i*0.01)
print()
print(f"Diametros (d): {Diametros_cm} cm")
print()

# 2.2- Para facilitar la construcción del inductor, se utilizó una separación entre espiras (S) del mismo tamaño que el Diametro del Conductor (D):
Se_cm = np.zeros(16)
for i in range(16):
    Se_cm[i] = Diametros_cm[i]
print(f"Separación entre espiras (S): {Se_cm} cm")
print()

# 2.3- Para realizar el inductor, se propone usar algún objeto lo más cilíndrico posible. Se elige el valor de 2.2 [cm] obtenido de un Slide de Guitarra:
# Vease: https://es.wikipedia.org/wiki/Slide_(guitarra)
Do_cm = 2.2
print(Do_cm)

# 2.4- Creo un vector con los posibles diámetros totales:
Dtotal_cm = np.zeros(16)
for i in range(16):
  Dtotal_cm[i]= Do_cm + 2* Diametros_cm[i]
print(f"Diámetro total del inductor (D_t): {Dtotal_cm} [cm]")
print()

# 2.5- Dado que se busca diseñar un inductor lo mas cuadrada posible, se plantea una relación de aspecto: l/d entre 1 y 2:
RelacionLD = np.zeros(11)
for i in range(11):
  RelacionLD[i] = 1 + i*0.1
print(f"Relaciones Diámetro-Largo (l/D): {RelacionLD} ")
print()

# 2.6- Dado que habrán 16 diámetros de cable posibles y con cada diámetro vamos a tener 11 opciones de largos para cada diámetro.
Largo_cm = np.zeros((11,16))                    # Array con los posibles largos (l).
for i in range(11):
  for j in range(16):
    Largo_cm[i,j] = Dtotal_cm[j] * RelacionLD[i]

N = np.zeros((11,16))                           # Array con número de espiras posibles (N).
for i in range(11):
  for j in range(16):
    N[i,j] = Largo_cm[i,j]/(2*Diametros_cm[j])



## 3- Tablas de parámetros
# 3.1- Se crearon las etiquetas para las filas y columnas y se usaron los valores reales, para que la tabla sea fácil de leer.
filas_relacion = [f"L/D={r:.1f}" for r in RelacionLD]
columnas_diams = [f"D_cu={d:.2f}cm" for d in Diametros_cm]

# 3.2- Se convertió la matriz de Largos en una Tabla (DataFrame):
df_largos = pd.DataFrame(Largo_cm, index=filas_relacion, columns=columnas_diams)

# 3.3 Convertir la matriz de Espiras (N) en una Tabla:
df_espiras = pd.DataFrame(N, index=filas_relacion, columns=columnas_diams)

# 3.4- Se muestran las tablas:
print("Tabla de Largos del Inductor [cm]")
display(df_largos)

print("\n Tabla del Número de Espiras (N)")
display(df_espiras)


Diametros (d): [0.08 0.09 0.1  0.11 0.12 0.13 0.14 0.15 0.16 0.17 0.18 0.19 0.2  0.21
 0.22 0.23] cm

Separación entre espiras (S): [0.08 0.09 0.1  0.11 0.12 0.13 0.14 0.15 0.16 0.17 0.18 0.19 0.2  0.21
 0.22 0.23] cm

2.2
Diámetro total del inductor (D_t): [2.36 2.38 2.4  2.42 2.44 2.46 2.48 2.5  2.52 2.54 2.56 2.58 2.6  2.62
 2.64 2.66] [cm]

Relaciones Diámetro-Largo (l/D): [1.  1.1 1.2 1.3 1.4 1.5 1.6 1.7 1.8 1.9 2. ] 

Tabla de Largos del Inductor [cm]


,D_cu=0.08cm,D_cu=0.09cm,D_cu=0.10cm,D_cu=0.11cm,D_cu=0.12cm,D_cu=0.13cm,D_cu=0.14cm,D_cu=0.15cm,D_cu=0.16cm,D_cu=0.17cm,D_cu=0.18cm,D_cu=0.19cm,D_cu=0.20cm,D_cu=0.21cm,D_cu=0.22cm,D_cu=0.23cm
L/D=1.0,2.360,2.380,2.40,2.420,2.440,2.460,2.480,2.50,2.520,2.540,2.560,2.580,2.60,2.620,2.640,2.660
L/D=1.1,2.596,2.618,2.64,2.662,2.684,2.706,2.728,2.75,2.772,2.794,2.816,2.838,2.86,2.882,2.904,2.926
L/D=1.2,2.832,2.856,2.88,2.904,2.928,2.952,2.976,3.00,3.024,3.048,3.072,3.096,3.12,3.144,3.168,3.192
L/D=1.3,3.068,3.094,3.12,3.146,3.172,3.198,3.224,3.25,3.276,3.302,3.328,3.354,3.38,3.406,3.432,3.458
L/D=1.4,3.304,3.332,3.36,3.388,3.416,3.444,3.472,3.50,3.528,3.556,3.584,3.612,3.64,3.668,3.696,3.724
L/D=1.5,3.540,3.570,3.60,3.630,3.660,3.690,3.720,3.75,3.780,3.810,3.840,3.870,3.90,3.930,3.960,3.990
L/D=1.6,3.776,3.808,3.84,3.872,3.904,3.936,3.968,4.00,4.032,4.064,4.096,4.128,4.16,4.192,4.224,4.256
L/D=1.7,4.012,4.046,4.08,4.114,4.148,4.182,4.216,4.25,4.284,4.318,4.352,4.386,4.42,4.454,4.488,4.522
L/D=1.8,4.248,4.284,4.32,4.356,4.392,4.428,4.464,4.50,4.536,4.572,4.608,4.644,4.68,4.716,4.752,4.788
L/D=1.9,4.484,4.522,4.56,4.598,4.636,4.674,4.712,4.75,4.788,4.826,4.864,4.902,4.94,4.978,5.016,5.054



 Tabla del Número de Espiras (N)


,D_cu=0.08cm,D_cu=0.09cm,D_cu=0.10cm,D_cu=0.11cm,D_cu=0.12cm,D_cu=0.13cm,D_cu=0.14cm,D_cu=0.15cm,D_cu=0.16cm,D_cu=0.17cm,D_cu=0.18cm,D_cu=0.19cm,D_cu=0.20cm,D_cu=0.21cm,D_cu=0.22cm,D_cu=0.23cm
L/D=1.0,14.750,13.222222,12.0,11.0,10.166667,9.461538,8.857143,8.333333,7.8750,7.470588,7.111111,6.789474,6.50,6.238095,6.0,5.782609
L/D=1.1,16.225,14.544444,13.2,12.1,11.183333,10.407692,9.742857,9.166667,8.6625,8.217647,7.822222,7.468421,7.15,6.861905,6.6,6.360870
L/D=1.2,17.700,15.866667,14.4,13.2,12.200000,11.353846,10.628571,10.000000,9.4500,8.964706,8.533333,8.147368,7.80,7.485714,7.2,6.939130
L/D=1.3,19.175,17.188889,15.6,14.3,13.216667,12.300000,11.514286,10.833333,10.2375,9.711765,9.244444,8.826316,8.45,8.109524,7.8,7.517391
L/D=1.4,20.650,18.511111,16.8,15.4,14.233333,13.246154,12.400000,11.666667,11.0250,10.458824,9.955556,9.505263,9.10,8.733333,8.4,8.095652
L/D=1.5,22.125,19.833333,18.0,16.5,15.250000,14.192308,13.285714,12.500000,11.8125,11.205882,10.666667,10.184211,9.75,9.357143,9.0,8.673913
L/D=1.6,23.600,21.155556,19.2,17.6,16.266667,15.138462,14.171429,13.333333,12.6000,11.952941,11.377778,10.863158,10.40,9.980952,9.6,9.252174
L/D=1.7,25.075,22.477778,20.4,18.7,17.283333,16.084615,15.057143,14.166667,13.3875,12.700000,12.088889,11.542105,11.05,10.604762,10.2,9.830435
L/D=1.8,26.550,23.800000,21.6,19.8,18.300000,17.030769,15.942857,15.000000,14.1750,13.447059,12.800000,12.221053,11.70,11.228571,10.8,10.408696
L/D=1.9,28.025,25.122222,22.8,20.9,19.316667,17.976923,16.828571,15.833333,14.9625,14.194118,13.511111,12.900000,12.35,11.852381,11.4,10.986957


Una vez obtenidas todas las posibles características constructivas físicas del Inductor:
 * Diametro del Cobre ($d$).
 * Diametro del Inductor ($D_{t}$).
 * Separación entre Espiras ($S$).
 * Largo del Inductor ($l$).
 * Relación de aspecto ($\frac{l}{D}$).

Podemos comenzar calculando primero el valor de inductancia del inductor, siguiendo la Fórmula de Wheeler para inductores monocapa con núcleo de aire (Véase: https://ingenio-triana.blogspot.com/2016/11/diseno-de-inductores-calculo-de-bobinas_2.html), siendo esta igual a:

$L = \frac{0,394 r^{2} N^{2}}{9r + 10l} [\mu Hy]$

Donde tanto $r$ como $l$, estarán en $[cm]$.

In [3]:
## 4- Se calcula el valor de las inductancias para cada caso siguiendo la formula de Wheeler para inductores monocapa con núcleo de aire:
# 4.1- Defino al radio del inductor como la mitad del diámetro:
r_cm = np.zeros(16)
for i in range(16):
  r_cm [i] = (Dtotal_cm[i] / 2)             # Basicamente: r = D_T/2 [cm]

# 4.2- Calculo la inductancia con la formula previamente vista:
inductancia_uh = np.zeros((11,16))
for i in range(11):
  for j in range(16):
    inductancia_uh [i,j] = (0.394 * (r_cm[j]**2) * (N[i,j]**2)) / (9 * r_cm[j] + 10 * Largo_cm[i,j])

# 4.3- Gráfico la tabla de inductancias de acuerdo a los Diámetros del Cobre (d) y la Relación de Aspecto (l/D):
df_inductancias = pd.DataFrame(inductancia_uh, index=filas_relacion, columns=columnas_diams)
print("Tabla de Inductancias del Inductor a diseñar [uH]")
display(df_inductancias)

Tabla de Inductancias del Inductor a diseñar [uH]


,D_cu=0.08cm,D_cu=0.09cm,D_cu=0.10cm,D_cu=0.11cm,D_cu=0.12cm,D_cu=0.13cm,D_cu=0.14cm,D_cu=0.15cm,D_cu=0.16cm,D_cu=0.17cm,D_cu=0.18cm,D_cu=0.19cm,D_cu=0.20cm,D_cu=0.21cm,D_cu=0.22cm,D_cu=0.23cm
L/D=1.0,3.487902,2.826533,2.347697,1.989157,1.713228,1.495984,1.321622,1.179358,1.061622,0.962967,0.879393,0.807905,0.746222,0.692586,0.645617,0.604223
L/D=1.1,3.948080,3.199453,2.657441,2.251597,1.939264,1.693357,1.495991,1.334957,1.201688,1.090017,0.995416,0.914496,0.844676,0.783962,0.730796,0.683942
L/D=1.2,4.413781,3.576849,2.970903,2.517187,2.168012,1.893099,1.672452,1.492424,1.343434,1.218591,1.112832,1.022367,0.944311,0.876436,0.816998,0.764617
L/D=1.3,4.884059,3.957954,3.287446,2.785388,2.399009,2.094805,1.850648,1.651438,1.486574,1.348429,1.231402,1.131297,1.044925,0.969818,0.904048,0.846085
L/D=1.4,5.358172,4.342166,3.606570,3.055775,2.631889,2.298154,2.030297,1.811749,1.630881,1.479326,1.350938,1.241116,1.146359,1.063961,0.991807,0.928218
L/D=1.5,5.835528,4.729007,3.927877,3.328012,2.866363,2.502896,2.211175,1.973157,1.776175,1.611118,1.471293,1.351687,1.248488,1.158749,1.080166,1.010912
L/D=1.6,6.315655,5.118093,4.251049,3.601829,3.102197,2.708825,2.393103,2.135501,1.922313,1.743675,1.592345,1.462899,1.351209,1.254087,1.169038,1.094086
L/D=1.7,6.798164,5.509110,4.575824,3.877005,3.339201,2.915777,2.575933,2.298651,2.069175,1.876890,1.713999,1.574663,1.454440,1.349898,1.258352,1.177673
L/D=1.8,7.282739,5.901801,4.901990,4.153359,3.577221,3.123614,2.759546,2.462500,2.216667,2.010676,1.836173,1.686905,1.558112,1.446119,1.348047,1.261618
L/D=1.9,7.769116,6.295952,5.229369,4.430740,3.816125,3.332224,2.943842,2.626958,2.364706,2.144958,1.958802,1.799565,1.662171,1.542698,1.438077,1.345875


Se procede ahora, a calcular las carácterísticas físicas de un inductor dadas sus inductancias, tales como:
 * Factor de Calidad Cargado: $Q_{C} = \frac{f_{o}}{BW} = 10 \ [\%]$ 
 * Factor de Calidad Descargado: $Q_{D} = 8550 \frac{D_{t} l}{102l + 450} \sqrt{f_{o}} \ \ $ ; $ \ \ $ con f en $[MHz]$ y $D, l$ en $[cm]$
 * Resistencia de Pérdida: $R_{P} = Q_{D} X_{L} = 2 \pi f_{o} L Q_{D} [\Omega]$ 

In [6]:
## 5- Cálculo de Factores de Calidad y Resistencias de Pérdida
# 5.1- Q Cargado: Este está basado en el BW entregado como requerimiento, y como el BW es el 10% de f_o, Q_c será constante para toda la matriz:
Q_cargado = f_o / (BW * 1e6)
print(f"El factor de calidad cargado es: {Q_cargado}")
print()

# 5.2- Q Descargado: Este se estima por frecuencia y geómetria, dada la fórmula mencionada arriba:
Q_descargado = np.zeros((11, 16))
for i in range(11):
    for j in range(16):
        Q_descargado[i, j] = 8550 * ((Dtotal_cm[j]*Largo_cm[i,j])/(120*Largo_cm[i,j]+450)) * np.sqrt(f_o_mhz)

# 5.3- Resistencia de Pérdidas: Este sigue la fórmula descripta arriba:
RP = np.zeros((11, 16))
for i in range(11):
    for j in range(16):
        # XL = 2 * pi * f * L
        XL = 2 * np.pi * f_o * (inductancia_uh[i, j] * 1e-6)
        RP[i, j] = Q_descargado[i, j] * XL

## 6- Generación de Tablas Finales

df_Qdescargado = pd.DataFrame(Q_descargado, index=filas_relacion, columns=columnas_diams)
df_RP = pd.DataFrame(RP, index=filas_relacion, columns=columnas_diams)


print("\nTabla del Factor de Calidad Descargado (Q_D)")
display(df_Qdescargado)

print("\nTabla de la Resistencia de Pérdida (R_P) en [Ohms]")
display(df_RP)

El factor de calidad cargado es: 10.0


Tabla del Factor de Calidad Descargado (Q_D)


,D_cu=0.08cm,D_cu=0.09cm,D_cu=0.10cm,D_cu=0.11cm,D_cu=0.12cm,D_cu=0.13cm,D_cu=0.14cm,D_cu=0.15cm,D_cu=0.16cm,D_cu=0.17cm,D_cu=0.18cm,D_cu=0.19cm,D_cu=0.20cm,D_cu=0.21cm,D_cu=0.22cm,D_cu=0.23cm
L/D=1.0,275.552222,279.328057,283.118657,286.923876,290.743574,294.577611,298.425848,302.288149,306.164380,310.054407,313.958100,317.875329,321.805966,325.749885,329.706961,333.677071
L/D=1.1,291.835248,295.777181,299.733601,303.704359,307.689306,311.688299,315.701194,319.727850,323.768127,327.821889,331.889000,335.969326,340.062735,344.169096,348.288282,352.420166
L/D=1.2,306.950606,311.041052,315.145654,319.264259,323.396716,327.542876,331.702593,335.875721,340.062119,344.261646,348.474162,352.699533,356.937622,361.188298,365.451428,369.726884
L/D=1.3,321.019551,325.243321,329.480882,333.732078,337.996755,342.274763,346.565951,350.870173,355.187284,359.517142,363.859607,368.214539,372.581803,376.961263,381.352789,385.756248
L/D=1.4,334.147109,338.491018,342.848331,347.218891,351.602542,355.999131,360.408508,364.830525,369.265035,373.711896,378.170965,382.642104,387.125175,391.620043,396.126575,400.644640
L/D=1.5,346.424707,350.877252,355.342804,359.821202,364.312292,368.815920,373.331934,377.860186,382.400530,386.952822,391.516920,396.092684,400.679978,405.278666,409.888615,414.509695
L/D=1.6,357.932304,362.483406,367.047112,371.623262,376.211701,380.812274,385.424832,390.049224,394.685307,399.332935,403.991968,408.662267,413.343694,418.036116,422.739400,427.453415
L/D=1.7,368.740135,373.380924,378.033915,382.698949,387.375871,392.064528,396.764769,401.476448,406.199418,410.933537,415.678665,420.434663,425.201396,429.978731,434.766536,439.564682
L/D=1.8,378.910144,383.632784,388.367228,393.113320,397.870904,402.639830,407.419948,412.211112,417.013178,421.826003,426.649450,431.483381,436.327663,441.182162,446.046750,450.921299
L/D=1.9,388.497175,393.294715,398.103671,402.923886,407.755208,412.597487,417.450574,422.314326,427.188599,432.073253,436.968152,441.873161,446.788146,451.712977,456.647527,461.591669



Tabla de la Resistencia de Pérdida (R_P) en [Ohms]


,D_cu=0.08cm,D_cu=0.09cm,D_cu=0.10cm,D_cu=0.11cm,D_cu=0.12cm,D_cu=0.13cm,D_cu=0.14cm,D_cu=0.15cm,D_cu=0.16cm,D_cu=0.17cm,D_cu=0.18cm,D_cu=0.19cm,D_cu=0.20cm,D_cu=0.21cm,D_cu=0.22cm,D_cu=0.23cm
L/D=1.0,108697.751951,89293.742072,75173.163056,64548.778344,56334.925226,49840.106579,44606.281187,40319.880802,36760.120759,33767.726882,31225.340921,29044.864150,27159.061271,25515.857404,24074.381932,22802.171787
L/D=1.1,130309.495763,107026.864620,90084.784002,77338.195933,67484.133237,59692.724350,53414.292495,48272.552081,44002.569271,40413.227983,37363.713140,34748.321302,32486.384501,30515.426253,28786.414020,27260.403151
L/D=1.2,153225.727036,125826.096829,105889.331846,90890.436861,79295.721066,70128.400765,62741.514469,56692.196037,51668.645199,47445.945744,43858.384343,40781.571856,38120.583468,35801.903353,33767.837003,31972.561365
L/D=1.3,177322.883988,145589.987126,122501.445571,105132.247519,91705.796395,81090.646566,72537.424549,65533.177693,59716.773569,54827.719066,50674.094869,47111.845518,44031.041752,41346.549993,38991.560264,36913.011267
L/D=1.4,202491.453611,166228.680099,139845.572219,119998.834643,104657.883626,92529.562752,82757.423487,74755.238460,68110.301043,62524.925793,57779.798535,53710.306079,50190.835201,47124.108867,44433.789504,42059.250583
L/D=1.5,228634.314675,187662.537769,157854.791575,135432.843680,118102.021495,104401.032806,93362.104055,84322.827639,76816.879684,70507.910491,65148.127871,60551.548820,56576.257161,53112.353571,50073.597528,47391.499479
L/D=1.6,255665.217558,209820.877800,176469.743092,151383.428187,131993.942979,116665.990224,104316.589358,94204.498844,85807.911738,78750.469963,72754.914275,67613.150412,63166.394132,59291.686709,55892.542041,52892.335755
L/D=1.7,283507.427014,232640.848970,195637.671239,167805.423366,146294.348187,129289.770174,115589.948113,104372.377490,95058.060562,87229.400734,80578.766623,74875.274244,69942.754225,65644.788515,61874.331982,58546.376794
L/D=1.8,312092.528757,256066.443769,215311.588425,184658.622379,160968.268555,142241.543321,127154.684702,114801.696975,104544.825172,95924.105917,88600.705613,82320.328364,76888.945533,72156.313045,68004.539351,64340.005207
L/D=1.9,341359.389116,280047.638674,235449.547010,201907.147700,175984.515146,155493.825091,138986.297647,125470.397532,114248.172685,104816.255915,96803.848514,89932.670885,83990.399412,78812.627046,74270.351761,70261.133475


Una vez calculado el inductor #L# y sabiendo el valor de la frecuencia de resonancia $f_{o}$, podemos definir a la capacidad total $C_{t}$ como:

$C_{t} = \frac{1}{(2 \pi f_{o})^2 L} \ [F]$

In [11]:
## 7- Calculo de C_t:
# 7.1- Calculo C_t con la fórmula descripta arriba:
C_total = np.zeros((11,16))
for i in range(11):
  for j in range(16):
    C_total[i,j] = 1/(((2*np.pi*f_o)**2)*inductancia_uh[i,j]*(1e-6))

# 7.2- Creo una tabla con todos los valores de C_t para cada posible Diámetro del Cobre (d) y Relación de Aspecto (l/D):
df_Ctotal = pd.DataFrame(C_total, index=filas_relacion, columns=columnas_diams)
print("\nTabla de Capacitancia Total (C_t) en [F]")
display(df_Ctotal)


Tabla de Capacitancia Total (C_t) en [F]


,D_cu=0.08cm,D_cu=0.09cm,D_cu=0.10cm,D_cu=0.11cm,D_cu=0.12cm,D_cu=0.13cm,D_cu=0.14cm,D_cu=0.15cm,D_cu=0.16cm,D_cu=0.17cm,D_cu=0.18cm,D_cu=0.19cm,D_cu=0.20cm,D_cu=0.21cm,D_cu=0.22cm,D_cu=0.23cm
L/D=1.0,2.241460e-11,2.765930e-11,3.330069e-11,3.930305e-11,4.563311e-11,5.225988e-11,5.915454e-11,6.629023e-11,7.364196e-11,8.118648e-11,8.890212e-11,9.676874e-11,1.047676e-10,1.128812e-10,1.210934e-10,1.293891e-10
L/D=1.1,1.980201e-11,2.443540e-11,2.941925e-11,3.472199e-11,4.031423e-11,4.616860e-11,5.225963e-11,5.856361e-11,6.505845e-11,7.172359e-11,7.853992e-11,8.548963e-11,9.255615e-11,9.972408e-11,1.069791e-10,1.143079e-10
L/D=1.2,1.771269e-11,2.185720e-11,2.631520e-11,3.105845e-11,3.606064e-11,4.129732e-11,4.674568e-11,5.238452e-11,5.819408e-11,6.415598e-11,7.025311e-11,7.646955e-11,8.279048e-11,8.920212e-11,9.569165e-11,1.022472e-10
L/D=1.3,1.600716e-11,1.975261e-11,2.378136e-11,2.806788e-11,3.258842e-11,3.732087e-11,4.224462e-11,4.734050e-11,5.259067e-11,5.797851e-11,6.348856e-11,6.910643e-11,7.481873e-11,8.061300e-11,8.647766e-11,9.240195e-11
L/D=1.4,1.459078e-11,1.800482e-11,2.167709e-11,2.558432e-11,2.970487e-11,3.401857e-11,3.850665e-11,4.315163e-11,4.793724e-11,5.284834e-11,5.787084e-11,6.299162e-11,6.819847e-11,7.348004e-11,7.882578e-11,8.422586e-11
L/D=1.5,1.339723e-11,1.653199e-11,1.990386e-11,2.349148e-11,2.727496e-11,3.123579e-11,3.535673e-11,3.962175e-11,4.401589e-11,4.852525e-11,5.313690e-11,5.783879e-11,6.261971e-11,6.746924e-11,7.237769e-11,7.733603e-11
L/D=1.6,1.237875e-11,1.527521e-11,1.839074e-11,2.170562e-11,2.520147e-11,2.886119e-11,3.266886e-11,3.660964e-11,4.066973e-11,4.483628e-11,4.909735e-11,5.344179e-11,5.785926e-11,6.234012e-11,6.687542e-11,7.145682e-11
L/D=1.7,1.150015e-11,1.419103e-11,1.708543e-11,2.016503e-11,2.341276e-11,2.681273e-11,3.035014e-11,3.401121e-11,3.778313e-11,4.165396e-11,4.561259e-11,4.964868e-11,5.375261e-11,5.791544e-11,6.212884e-11,6.638507e-11
L/D=1.8,1.073496e-11,1.324679e-11,1.594861e-11,1.882330e-11,2.185494e-11,2.502868e-11,2.833072e-11,3.174819e-11,3.526914e-11,3.888241e-11,4.257764e-11,4.634518e-11,5.017605e-11,5.406189e-11,5.799494e-11,6.196797e-11
L/D=1.9,1.006291e-11,1.241749e-11,1.495016e-11,1.764489e-11,2.048673e-11,2.346179e-11,2.655710e-11,2.976063e-11,3.306115e-11,3.644822e-11,3.991212e-11,4.344380e-11,4.703483e-11,5.067741e-11,5.436423e-11,5.808854e-11
